In [1]:
import pandas as pd
import glob
import tqdm
from collections import Counter
import json

In [2]:
files_codegeneration = glob.glob("/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/**/codegeneration_**", recursive=True)
files_eval_all = glob.glob("/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/**/eval_all_codegeneration_**", recursive=True)
files_eval = glob.glob("/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/**/eval_codegeneration_**", recursive=True)


In [3]:
def fuck_id(s):
    name = s.split('/')[-1]
    if name.startswith("codegeneration_"):
        name = '/'.join(name.split('_')[1:])
    elif name.startswith("eval_all_codegeneration_"):
        name = '/'.join(name.split('_')[3:])
    elif name.startswith("eval_codegeneration_"):
        name = '/'.join(name.split('_')[2:])
    else:
        raise
    return "/".join(s.split('/')[:-1]) + "/" + name
a = set([fuck_id(i) for i in files_codegeneration])
b = set([fuck_id(i) for i in files_eval_all])
c = set([fuck_id(i) for i in files_eval])
len(a), len(b), len(c), len(a&b&c)

(4554, 4235, 4235, 4163)

In [4]:
# files = 
files = [i for i in files_codegeneration if fuck_id(i) in (a&b&c)]

In [5]:
assert all([i.count('codegeneration_') == 1 for i in files])

In [6]:
def get_folder(s):
    return '/'.join(s.split('/')[:-1])

def get_upfix(s):
    return s.split('/')[-1][len('codegeneration_'):]

def get_model(s):
    return s.split('/')[-2]

def get_language(s):
    return get_upfix(s).split('_')[0]

def get_params(s):
    res = '_'.join(get_upfix(s).split('_')[1:])
    cot  = False
    if res.endswith("_cot copy.json"):
        cot = True
        res = res[:-len("_cot copy.json")]
    if res.endswith("_cot.json"):
        cot = True
        res = res[:-len("_cot.json")]
    elif res.endswith(".json"):
        res = res[:-len(".json")]
        
    n, T, topk = res.split('_')
    return {"T": T,
            "n": n,
            "top_k": topk,
            "cot": cot
           }

def get_all(s): 
    return {**{"folder": get_folder(s),
            "upfix": get_upfix(s),
            "model": get_model(s),
            "language": get_language(s)
    }, **get_params(s)}
             
get_all(files[10])

{'folder': '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/output_v1_250923 copy/DeepSeek-R1-0528_cot',
 'upfix': 'scala_1_0.2_0.95_cot.json',
 'model': 'DeepSeek-R1-0528_cot',
 'language': 'scala',
 'T': '0.2',
 'n': '1',
 'top_k': '0.95',
 'cot': True}

In [9]:
data = []
dct_orig = None
for _fn in tqdm.tqdm(files):
    fn = _fn.replace("codegeneration_", "eval_codegeneration_")
    dct_orig = json.load(open(fn, "r"))
    if not dct_orig:
        print("empty %s"%fn)
        continue
    
    try:
        dct_orig[0]['pass@1']
        dct_orig[1]
    except:
        print("failed to fetch %s"%fn)
        raise
        continue

    dct = {'fn': _fn,
           "pass@1": dct_orig[0]['pass@1'],
           "n_rows": len(dct_orig[1])
          }
    dct.update(get_all(_fn))
    data.append(dct)

100%|██████████| 4163/4163 [05:50<00:00, 11.89it/s]


In [11]:
# data[70]['pass@1']

0.3090047393364929

In [14]:
# data['pass@1']
data_orig = data

In [12]:
# dct_orig[2]['pass@1']
# json.load(open("/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/output_v1_250923 copy/DeepSeek-R1-0528_cot/eval_codegeneration_c#_1_0.2_0.95_cot.json",
#                "r"
#               )
#          )[0]['pass@1']




In [15]:
data = pd.DataFrame(data_orig)

In [ ]:
data['T'] = data['T'].apply(float)
data['n'] = data['n'].apply(int)
data['top_k'] = data['top_k'].apply(float)
data['cot'] = data['cot'].apply(bool)

In [140]:
# Удаляем дубликат
data = data[data['fn']!='/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/tmp/DeepSeek-R1-0528_cot_old/codegeneration_kotlin_1_0.2_0.95_cot copy.json']
data = data[data['fn'] != '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_2502_2505/output/DeepSeek-R1-0528_old_w/codegeneration_c++_1_0.2_0.95.json']
data.reset_index(drop=True, inplace=True)

In [143]:
# data[(data['folder']=='/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_2502_2505/output/DeepSeek-R1-0528_old_w') & \
#      (data['upfix'] == 'c++_1_0.2_0.95.json')
#     ]



In [142]:
# data[(data['model_name'] == 'DeepSeek-R1-0528') &\
#      (data['language'] == 'c++')]

In [144]:
# data.loc[956]['fn']

In [145]:
# data÷
# data

In [146]:
rename_dict = {
    # DeepSeek-R1-0528
    "DeepSeek-R1-0528 copy": "DeepSeek-R1-0528",
    "DeepSeek-R1-0528": "DeepSeek-R1-0528",
    "DeepSeek-R1-0528_cot copy": "DeepSeek-R1-0528",
    "DeepSeek-R1-0528_cot": "DeepSeek-R1-0528",
    "DeepSeek-R1-0528_cot_old": "DeepSeek-R1-0528",
    "DeepSeek-R1-0528_old_w": "DeepSeek-R1-0528",

    # DeepSeek-R1-0528-Qwen3-8B (deepseek-ai/DeepSeek-R1-0528-Qwen3-8B)
    "DeepSeek-R1-0528-Qwen3-8B_cot copy": "DeepSeek-R1-0528-Qwen3-8B",
    "DeepSeek-R1-0528-Qwen3-8B_cot": "DeepSeek-R1-0528-Qwen3-8B",

    # DeepSeek-R1-Distill-Qwen-14B (deepseek-ai/DeepSeek-R1-Distill-Qwen-14B)
    "DeepSeek-R1-Distill-Qwen-14B_cot copy 2": "DeepSeek-R1-Distill-Qwen-14B",
    "DeepSeek-R1-Distill-Qwen-14B_cot copy": "DeepSeek-R1-Distill-Qwen-14B",
    "DeepSeek-R1-Distill-Qwen-14B_cot": "DeepSeek-R1-Distill-Qwen-14B",

    # DeepSeek-R1-Distill-Qwen-32B (deepseek-ai/DeepSeek-R1-Distill-Qwen-32B)
    "DeepSeek-R1-Distill-Qwen-32B_cot copy 2": "DeepSeek-R1-Distill-Qwen-32B",
    "DeepSeek-R1-Distill-Qwen-32B_cot copy": "DeepSeek-R1-Distill-Qwen-32B",
    "DeepSeek-R1-Distill-Qwen-32B_cot": "DeepSeek-R1-Distill-Qwen-32B",

    # DeepSeek-R1 (deepseek-ai/DeepSeek-R1)
    "DeepSeek-R1_cot": "DeepSeek-R1",

    # DeepSeek-V3-0324 (deepseek-ai/DeepSeek-V3-0324) — "V3.1" скорее всего DeepSeek-V3-0324
    "DeepSeek-V3.1_cot": "DeepSeek-V3-0324",
    "V3.1": "DeepSeek-V3-0324",
    "V3.1_cot": "DeepSeek-V3-0324",

    # Devstral-Small-2505 (mistralai/Devstral-Small-2505)
    "Devstral-Small-2505": "Devstral-Small-2505",
    "Devstral-Small-2505_cot copy 2": "Devstral-Small-2505",
    "Devstral-Small-2505_cot copy": "Devstral-Small-2505",
    "Devstral-Small-2505_cot": "Devstral-Small-2505",

    # O4-Mini — OpenAI модель, нет на HuggingFace
    "O4-Mini (Medium)_cot": "o4-mini",

    # OlympicCoder-32B (open-r1/OlympicCoder-32B)
    "OlympicCoder-32B_cot copy 2": "OlympicCoder-32B",
    "OlympicCoder-32B_cot copy": "OlympicCoder-32B",
    "OlympicCoder-32B_cot": "OlympicCoder-32B",

    # OlympicCoder-7B (open-r1/OlympicCoder-7B)
    "OlympicCoder-7B_cot copy 2": "OlympicCoder-7B",
    "OlympicCoder-7B_cot copy": "OlympicCoder-7B",
    "OlympicCoder-7B_cot": "OlympicCoder-7B",

    # OpenCodeReasoning-Nemotron-32B (nvidia/OpenCodeReasoning-Nemotron-1.1-32B)
    "OpenCodeReasoning-Nemotron-1.1-32B_cot copy 2": "OpenCodeReasoning-Nemotron-1.1-32B",
    "OpenCodeReasoning-Nemotron-1.1-32B_cot copy": "OpenCodeReasoning-Nemotron-1.1-32B",
    "OpenCodeReasoning-Nemotron-1.1-32B_cot": "OpenCodeReasoning-Nemotron-1.1-32B",

    # OpenCoder-8B-Instruct (infly/OpenCoder-8B-Instruct)
    "OpenCoder-8B-Instruct copy 2": "OpenCoder-8B-Instruct",
    "OpenCoder-8B-Instruct copy": "OpenCoder-8B-Instruct",
    "OpenCoder-8B-Instruct": "OpenCoder-8B-Instruct",

    # OpenReasoning-Nemotron-32B (nvidia/OpenReasoning-Nemotron-32B)
    "OpenReasoning-Nemotron-32B_cot copy 2": "OpenReasoning-Nemotron-32B",
    "OpenReasoning-Nemotron-32B_cot copy": "OpenReasoning-Nemotron-32B",
    "OpenReasoning-Nemotron-32B_cot": "OpenReasoning-Nemotron-32B",

    # Qwen2.5-Coder-14B-Instruct (Qwen/Qwen2.5-Coder-14B-Instruct)
    "Qwen2.5-Coder-14B-Instruct copy 2": "Qwen2.5-Coder-14B-Instruct",
    "Qwen2.5-Coder-14B-Instruct copy": "Qwen2.5-Coder-14B-Instruct",
    "Qwen2.5-Coder-14B-Instruct": "Qwen2.5-Coder-14B-Instruct",

    # Qwen2.5-Coder-32B-Instruct (Qwen/Qwen2.5-Coder-32B-Instruct)
    "Qwen2.5-Coder-32B-Instruct copy 2": "Qwen2.5-Coder-32B-Instruct",
    "Qwen2.5-Coder-32B-Instruct copy": "Qwen2.5-Coder-32B-Instruct",
    "Qwen2.5-Coder-32B-Instruct": "Qwen2.5-Coder-32B-Instruct",
    "Qwen2.5-Coder-32B-Instruct_cot": "Qwen2.5-Coder-32B-Instruct",

    # Qwen3-14B (Qwen/Qwen3-14B)
    "Qwen3-14B_cot copy 2": "Qwen3-14B",
    "Qwen3-14B_cot copy": "Qwen3-14B",
    "Qwen3-14B_cot": "Qwen3-14B",

    # Qwen3-235B-A22B-Instruct-2507 (Qwen/Qwen3-235B-A22B-Instruct-2507)
    "Qwen3-235B-A22B-Instruct-2507 copy 2": "Qwen3-235B-A22B-Instruct-2507",
    "Qwen3-235B-A22B-Instruct-2507 copy": "Qwen3-235B-A22B-Instruct-2507",
    "Qwen3-235B-A22B-Instruct-2507-FP8_cot": "Qwen3-235B-A22B-Instruct-2507",
    "Qwen3-235B-A22B-Instruct-2507": "Qwen3-235B-A22B-Instruct-2507",

    # Qwen3-235B-A22B (Qwen/Qwen3-235B-A22B) — Thinking-2507 = thinking mode
    "Qwen3-235B-A22B-Thinking-2507-FP8_cot": "Qwen3-235B-A22B-Instruct-2507",
    "Qwen3-235B-A22B-Thinking-2507_cot copy 2": "Qwen3-235B-A22B-Instruct-2507",
    "Qwen3-235B-A22B-Thinking-2507_cot copy": "Qwen3-235B-A22B-Instruct-2507",
    "Qwen3-235B-A22B-Thinking-2507_cot": "Qwen3-235B-A22B-Instruct-2507",

    # Qwen3-235B-A22B (Qwen/Qwen3-235B-A22B) — до 2507
    "Qwen3-235B-A22B_cot copy 2": "Qwen3-235B-A22B",
    "Qwen3-235B-A22B_cot copy": "Qwen3-235B-A22B",
    "Qwen3-235B-A22B_cot": "Qwen3-235B-A22B",

    # Qwen3-30B-A3B-Instruct-2507 (Qwen/Qwen3-30B-A3B-Instruct-2507)
    "Qwen3-30B-A3B-Instruct-2507 copy 2": "Qwen3-30B-A3B-Instruct-2507",
    "Qwen3-30B-A3B-Instruct-2507 copy": "Qwen3-30B-A3B-Instruct-2507",
    "Qwen3-30B-A3B-Instruct-2507": "Qwen3-30B-A3B-Instruct-2507",

    # Qwen3-30B-A3B Thinking-2507 = thinking mode of Instruct-2507
    "Qwen3-30B-A3B-Thinking-2507_cot copy 2": "Qwen3-30B-A3B-Instruct-2507",
    "Qwen3-30B-A3B-Thinking-2507_cot copy": "Qwen3-30B-A3B-Instruct-2507",
    "Qwen3-30B-A3B-Thinking-2507_cot": "Qwen3-30B-A3B-Instruct-2507",

    # Qwen3-30B-A3B (Qwen/Qwen3-30B-A3B) — до 2507
    "Qwen3-30B-A3B_cot copy 2": "Qwen3-30B-A3B",
    "Qwen3-30B-A3B_cot copy": "Qwen3-30B-A3B",
    "Qwen3-30B-A3B_cot": "Qwen3-30B-A3B",

    # Qwen3-32B (Qwen/Qwen3-32B)
    "Qwen3-32B_cot copy 2": "Qwen3-32B",
    "Qwen3-32B_cot copy": "Qwen3-32B",
    "Qwen3-32B_cot": "Qwen3-32B",

    # Qwen3-8B (Qwen/Qwen3-8B)
    "Qwen3-8B_cot copy 2": "Qwen3-8B",
    "Qwen3-8B_cot copy": "Qwen3-8B",
    "Qwen3-8B_cot": "Qwen3-8B",

    # Qwen3-Coder-30B-A3B-Instruct (Qwen/Qwen3-Coder-30B-A3B-Instruct)
    "Qwen3-Coder-30B-A3B-Instruct copy 2": "Qwen3-Coder-30B-A3B-Instruct",
    "Qwen3-Coder-30B-A3B-Instruct copy": "Qwen3-Coder-30B-A3B-Instruct",
    "Qwen3-Coder-30B-A3B-Instruct": "Qwen3-Coder-30B-A3B-Instruct",

    # Seed-Coder-8B-Instruct (ByteDance-Seed/Seed-Coder-8B-Instruct)
    "Seed-Coder-8B-Instruct copy 2": "Seed-Coder-8B-Instruct",
    "Seed-Coder-8B-Instruct copy": "Seed-Coder-8B-Instruct",
    "Seed-Coder-8B-Instruct": "Seed-Coder-8B-Instruct",

    # Seed-Coder-8B-Reasoning (ByteDance-Seed/Seed-Coder-8B-Reasoning)
    "Seed-Coder-8B-Reasoning_cot copy 2": "Seed-Coder-8B-Reasoning",
    "Seed-Coder-8B-Reasoning_cot copy": "Seed-Coder-8B-Reasoning",
    "Seed-Coder-8B-Reasoning_cot": "Seed-Coder-8B-Reasoning",

    # deepseek-coder-33b-instruct (deepseek-ai/deepseek-coder-33b-instruct)
    "deepseek-coder-33b-instruct copy 2": "deepseek-coder-33b-instruct",
    "deepseek-coder-33b-instruct copy": "deepseek-coder-33b-instruct",
    "deepseek-coder-33b-instruct": "deepseek-coder-33b-instruct",

    # === Неизвестные / внутренние модели ===
    "_cot": "unknown_empty",

    # giga* — внутренние модели (нет на HuggingFace)
    "giga38b_thinking_v1_giga_base_save_n_epoch_3_cot": "giga38b_giga_base_epoch_3",
    "giga38b_thinking_v1_giga_base_save_n_epoch_4_cot": "giga38b_giga_base_epoch_4",
    "giga38b_thinking_v1_qwen25coder32_base_32_save_n_epoch_4_cot": "giga38b_qwen25coder32_base_epoch_4",
    "giga9b_thinking_v6.5_epoch_0": "giga9b_v6.5_epoch_0",
    "giga9b_thinking_v6.5_epoch_1": "giga9b_v6.5_epoch_1",
    "giga9b_thinking_v6.5_epoch_2": "giga9b_v6.5_epoch_2",
    "giga9b_thinking_v6.5_epoch_3": "giga9b_v6.5_epoch_3",
    "giga9b_thinking_v6.5_epoch_4": "giga9b_v6.5_epoch_4",

    # global_step_* — внутренние чекпоинты
    "global_step_12_cont_cot": "unknown_step_12_cont",
    "global_step_21_cot": "unknown_step_21",
    "global_step_27_cot": "unknown_step_27",
    "global_step_60_cot": "unknown_step_60",
    "global_step_66_cot": "unknown_step_66",

    # gpt-oss-* — внутренние модели
    "gpt-oss-120b_cot": "gpt-oss-120b",
    "gpt-oss-120b_cot_old_w": "gpt-oss-120b",
    "gpt-oss-20b_cot": "gpt-oss-20b",
    "gpt-oss-20b_cot_old": "gpt-oss-20b",
}


In [147]:
data['model_name'] = data['model'].apply(lambda x: rename_dict[x])

In [148]:
def get_exp_name(row):
    return f"{row['folder']}_{row['model']}_{row['n']}_{row['T']}_{row['top_k']}_{row['cot']}"
data['exp_name']  = data.apply(get_exp_name, axis = 1)

In [149]:
# data['pass@1'].max()
# data

In [150]:
data

,fn,pass@1,n_rows,folder,upfix,model,language,T,n,top_k,cot,model_name,exp_name
0,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,0.557252,131,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,c#_1_0.2_0.95_cot.json,DeepSeek-R1-0528_cot,c#,0.2,1,0.95,True,DeepSeek-R1-0528,/workspace-SR008.fs2/nfs2_migration/rlevichev/...
1,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,0.641221,131,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,c++_1_0.2_0.95_cot.json,DeepSeek-R1-0528_cot,c++,0.2,1,0.95,True,DeepSeek-R1-0528,/workspace-SR008.fs2/nfs2_migration/rlevichev/...
2,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,0.473282,131,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,go_1_0.2_0.95_cot.json,DeepSeek-R1-0528_cot,go,0.2,1,0.95,True,DeepSeek-R1-0528,/workspace-SR008.fs2/nfs2_migration/rlevichev/...
3,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,0.641221,131,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,java_1_0.2_0.95_cot.json,DeepSeek-R1-0528_cot,java,0.2,1,0.95,True,DeepSeek-R1-0528,/workspace-SR008.fs2/nfs2_migration/rlevichev/...
4,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,0.557252,131,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,javascript_1_0.2_0.95_cot.json,DeepSeek-R1-0528_cot,javascript,0.2,1,0.95,True,DeepSeek-R1-0528,/workspace-SR008.fs2/nfs2_migration/rlevichev/...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4156,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,0.220284,1055,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,python_10_0.2_0.95.json,deepseek-coder-33b-instruct,python,0.2,10,0.95,False,deepseek-coder-33b-instruct,/workspace-SR008.fs2/nfs2_migration/rlevichev/...
4157,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,0.187299,1055,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,ruby_10_0.2_0.95.json,deepseek-coder-33b-instruct,ruby,0.2,10,0.95,False,deepseek-coder-33b-instruct,/workspace-SR008.fs2/nfs2_migration/rlevichev/...
4158,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,0.027583,1055,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,rust_10_0.2_0.95.json,deepseek-coder-33b-instruct,rust,0.2,10,0.95,False,deepseek-coder-33b-instruct,/workspace-SR008.fs2/nfs2_migration/rlevichev/...
4159,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,0.148152,1055,/workspace-SR008.fs2/nfs2_migration/rlevichev/...,scala_10_0.2_0.95.json,deepseek-coder-33b-instruct,scala,0.2,10,0.95,False,deepseek-coder-33b-instruct,/workspace-SR008.fs2/nfs2_migration/rlevichev/...


In [155]:
def foo(df):
    assert df['n_rows'].nunique() == 1
    assert df['language'].nunique() == len(df)
    assert df['n_rows'].nunique() == 1
    row = df.iloc[0]
    return pd.Series({"model_name": row['model_name'],
                      "n_rows": row['n_rows'],
                      "n": row['n'],
                      "T": row['T'],
                      "top_k": row['top_k'],
                      "cot": row['cot'],
                      "pass@1": df['pass@1'].mean(),
                      "n_language": df['language'].nunique()
                    })
df_stat = data.groupby('exp_name').apply(foo)

/tmp/ipykernel_1417199/1194467877.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_stat = data.groupby('exp_name').apply(foo)


In [152]:
data[data['model_name']=='gpt-oss-20b']['pass@1'].max()

np.float64(0.5648854961832062)

In [153]:
data[data['model_name'].apply(lambda x: 'oss' in x.lower())]['model_name'].unique()

array(['gpt-oss-120b', 'gpt-oss-20b'], dtype=object)

In [163]:
a = df_stat.sort_values(["n_rows", "n_language", "pass@1"], ascending = False).drop_duplicates("model_name").dropna()

a.reset_index(drop=True).\
sort_values("pass@1", ascending = False).shape

(41, 8)

In [165]:
a.reset_index().sort_values("pass@1", ascending = False)['exp_name'].unique()

array(['/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Qwen3-32B_cot_Qwen3-32B_cot_10_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/Qwen3-235B-A22B-Thinking-2507_cot_Qwen3-235B-A22B-Thinking-2507_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v2_more_lang/livecodebench-mult-feature-add_php_scala_kotlin/output/O4-Mini (Medium)_cot_O4-Mini (Medium)_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Qwen3-30B-A3B_cot_Qwen3-30B-A3B_cot_10_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/

In [166]:
good_exp_name = ['/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Qwen3-32B_cot_Qwen3-32B_cot_10_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/Qwen3-235B-A22B-Thinking-2507_cot_Qwen3-235B-A22B-Thinking-2507_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v2_more_lang/livecodebench-mult-feature-add_php_scala_kotlin/output/O4-Mini (Medium)_cot_O4-Mini (Medium)_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Qwen3-30B-A3B_cot_Qwen3-30B-A3B_cot_10_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Qwen3-14B_cot_Qwen3-14B_cot_10_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/Qwen3-30B-A3B-Thinking-2507_cot_Qwen3-30B-A3B-Thinking-2507_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/Qwen3-235B-A22B_cot_Qwen3-235B-A22B_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev/output/DeepSeek-R1-0528_DeepSeek-R1-0528_1_0.2_0.95_False',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/test/output_32k/output/giga38b_thinking_v1_qwen25coder32_base_32_save_n_epoch_4_cot_giga38b_thinking_v1_qwen25coder32_base_32_save_n_epoch_4_cot_10_0.6_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/test/output_32k/output/giga38b_thinking_v1_giga_base_save_n_epoch_3_cot_giga38b_thinking_v1_giga_base_save_n_epoch_3_cot_10_0.6_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/test/output_32k/output/giga38b_thinking_v1_giga_base_save_n_epoch_4_cot_giga38b_thinking_v1_giga_base_save_n_epoch_4_cot_10_0.6_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Qwen3-8B_cot_Qwen3-8B_cot_10_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v2_more_lang/livecodebench-mult-feature-add_php_scala_kotlin/output/DeepSeek-R1_cot_DeepSeek-R1_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_2502_2505/output/DeepSeek-R1-0528-Qwen3-8B_cot_DeepSeek-R1-0528-Qwen3-8B_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/OlympicCoder-32B_cot_OlympicCoder-32B_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_2502_2505/output/gpt-oss-20b_cot_gpt-oss-20b_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Qwen3-Coder-30B-A3B-Instruct_Qwen3-Coder-30B-A3B-Instruct_10_0.2_0.95_False',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev/output/V3.1_cot_V3.1_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Qwen2.5-Coder-32B-Instruct_Qwen2.5-Coder-32B-Instruct_10_0.2_0.95_False',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Qwen2.5-Coder-14B-Instruct_Qwen2.5-Coder-14B-Instruct_10_0.2_0.95_False',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/OlympicCoder-7B_cot_OlympicCoder-7B_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Seed-Coder-8B-Instruct_Seed-Coder-8B-Instruct_10_0.2_0.95_False',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/DeepSeek-R1-Distill-Qwen-32B_cot_DeepSeek-R1-Distill-Qwen-32B_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/OpenReasoning-Nemotron-32B_cot_OpenReasoning-Nemotron-32B_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/Devstral-Small-2505_cot_Devstral-Small-2505_cot_10_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/OpenCodeReasoning-Nemotron-1.1-32B_cot_OpenCodeReasoning-Nemotron-1.1-32B_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/DeepSeek-R1-Distill-Qwen-14B_cot_DeepSeek-R1-Distill-Qwen-14B_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/giga9b_thinking_v6.5_epoch_4_giga9b_thinking_v6.5_epoch_4_10_0.6_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/giga9b_thinking_v6.5_epoch_3_giga9b_thinking_v6.5_epoch_3_10_0.6_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/giga9b_thinking_v6.5_epoch_2_giga9b_thinking_v6.5_epoch_2_10_0.6_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts_full/output/deepseek-coder-33b-instruct_deepseek-coder-33b-instruct_10_0.2_0.95_False',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/giga9b_thinking_v6.5_epoch_1_giga9b_thinking_v6.5_epoch_1_10_0.6_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v2_more_lang/livecodebench-mult-feature-add_php_scala_kotlin/output/gpt-oss-120b_cot_gpt-oss-120b_cot_1_0.2_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/giga9b_thinking_v6.5_epoch_0_giga9b_thinking_v6.5_epoch_0_10_0.6_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v1/livecodebench-mult-dev_merge/output/OpenCoder-8B-Instruct_OpenCoder-8B-Instruct_1_0.2_0.95_False',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/_cot__cot_10_1.0_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/global_step_21_cot_global_step_21_cot_10_1.0_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/global_step_12_cont_cot_global_step_12_cont_cot_10_1.0_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/global_step_27_cot_global_step_27_cot_10_1.0_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/global_step_66_cot_global_step_66_cot_10_1.0_0.95_True',
       '/workspace-SR008.fs2/nfs2_migration/rlevichev/projects/test_lcb/final_versions/v3/livecodebench-mult-feature-configurable-system-prompts/output/global_step_60_cot_global_step_60_cot_10_1.0_0.95_True']

In [167]:
df = data[data['exp_name'].isin(good_exp_name)]

In [132]:
from functools import cache

@cache
def process_results(folder, upfix):
    """
    Loads 3 JSON files and returns pass@1 metric and per-task details.
    
    Args:
        folder (str): folder with files
        upfix (str): file suffix
    
    Returns:
        dict with:
            - pass@1 (float): overall pass@1 metric
            - tasks (List[Dict]): per-task details
    """
    import json
    from pathlib import Path
    
    folder_path = Path(folder)
    
    with open(folder_path / f"codegeneration_{upfix}", "r") as f:
        codegeneration = json.load(f)
    
    with open(folder_path / f"eval_codegeneration_{upfix}", "r") as f:
        eval_data = json.load(f)
    
    with open(folder_path / f"eval_all_codegeneration_{upfix}", "r") as f:
        eval_all = json.load(f)
    
    # eval_data has 3 items (one per eval config?), each with aggregate pass@1
    # eval_data[0]["detail"] likely has per-task breakdown
    # Use first eval_data entry as primary
    overall_pass1 = eval_data[0]["pass@1"]
    eval_detail = eval_data[0]["detail"]  # list of per-task detail dicts
    
    tasks = []
    for i in range(len(eval_all)):
        ea = eval_all[i]
        cg = codegeneration[i]# if i < len(codegeneration) else {}
        
        # eval_detail could be a list (per-task) or a dict
        if isinstance(eval_detail, list):# and i < len(eval_detail):
            ed = eval_detail[i]
        elif isinstance(eval_detail, dict):
            ed = eval_detail
        else:
            ed = {}
        
        task = {
            # metadata
            "question_title": ea["question_title"],
            "question_content": ea["question_content"],
            "platform": ea["platform"],
            "question_id": ea["question_id"],
            "contest_id": ea["contest_id"],
            "contest_date": ea["contest_date"],
            "starter_code": ea["starter_code"],
            "difficulty": ea["difficulty"],
            # generation
            "output_list": ea["output_list"],
            "code_list": ea["code_list"],
            "prompt": ea["prompt"],
            # eval from eval_all
            "graded_list": ea["graded_list"],
            "pass@1": ea["pass@1"],
            "metadata": ea["metadata"],
            # eval detail from eval_codegeneration
            "eval_detail": ed,
        }
        tasks.append(task)
    
    return {
        "pass@1": overall_pass1,
        "tasks": tasks,
    }


In [168]:
assert df.shape[0] == df[['folder', 'upfix']].drop_duplicates().shape[0]

In [169]:
for _, row in tqdm.tqdm(df.iterrows(), total = len(df)):
    try:
        process_results(row['folder'], row['upfix'])
    except IndexError:
        print(row['folder'], row['upfix'])

100%|██████████| 430/430 [03:41<00:00,  1.94it/s] 


In [178]:
!rm -rf /home/jovyan/adamenko/multilcb_data/data/

In [179]:
!mkdir /home/jovyan/adamenko/multilcb_data/data/

In [180]:
import os

def fetch_data(folder, upfix):
   tasks = process_results(folder, upfix)
   tasks = tasks['tasks']

   def normalize(dct):
       return {'date': dct['contest_date'], 
               'task_id': dct['question_id'], 
               'pass': dct['pass@1'], 
               'platform': dct['platform'], 
               'difficulty': dct['difficulty'][0].upper()
              }

   return [normalize(t) for t in tasks]

for ind, row in df.iterrows():
    df_task = pd.DataFrame(fetch_data(row['folder'], row['upfix']))
    folder = f"/home/jovyan/adamenko/multilcb_data/data/{row['model_name']}{'_cot' if row['cot'] else ''}"
    
    rename_dict = {
        "c#": "csharp", "c++": "cpp",
        "go": "go", "java": "java",
        "javascript": "javascript",
        "python": "python",
        "ruby": "ruby", "rust": "rust",
        "typescript": "typescript", 
        "kotlin": "kotlin",
        "php": "php", "scala": "scala",
    }
    
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    fn = f"{folder}/{rename_dict[row['language']]}{'_cot' if row['cot'] else ''}.jsonl"
    pd.DataFrame(fetch_data(row['folder'], row['upfix'])).to_json(fn, lines=True, orient='records')
    
    meta_data = {"temperature": float(row['T']),
                 "top_k": float(row['top_k']),
                 "reasoning": bool(row['cot']),
                 "pass_at_n": int(row['n'])
                }
    
    fn_meta = f"{folder}/meta.json"
    if os.path.exists(fn_meta):
        meta_old = json.load(open(f"{folder}/meta.json", "r"))
        for k in meta_old:
            assert meta_old[k] == meta_data[k], (k, fn_meta)
    else:
        json.dump(meta_data, open(f"{folder}/meta.json", "w+"))

In [181]:
# df[df['model_name']=='_cot']

In [182]:
# !ls ."

In [183]:
!zip -r data.zip data

  adding: data/ (stored 0%)
  adding: data/giga38b_giga_base_epoch_3_cot/ (stored 0%)
  adding: data/giga38b_giga_base_epoch_3_cot/typescript_cot.jsonl (deflated 93%)
  adding: data/giga38b_giga_base_epoch_3_cot/meta.json (deflated 8%)
  adding: data/giga38b_giga_base_epoch_4_cot/ (stored 0%)
  adding: data/giga38b_giga_base_epoch_4_cot/typescript_cot.jsonl (deflated 93%)
  adding: data/giga38b_giga_base_epoch_4_cot/meta.json (deflated 8%)
  adding: data/giga38b_qwen25coder32_base_epoch_4_cot/ (stored 0%)
  adding: data/giga38b_qwen25coder32_base_epoch_4_cot/typescript_cot.jsonl (deflated 93%)
  adding: data/giga38b_qwen25coder32_base_epoch_4_cot/meta.json (deflated 8%)
  adding: data/DeepSeek-R1-0528/ (stored 0%)
  adding: data/DeepSeek-R1-0528/csharp.jsonl (deflated 94%)
  adding: data/DeepSeek-R1-0528/meta.json (deflated 8%)
  adding: data/DeepSeek-R1-0528/cpp.jsonl (deflated 94%)
  adding: data/DeepSeek-R1-0528/go.jsonl (deflated 94%)
  adding: data/DeepSeek-R1-0528/java.jsonl (def

## Trash bellow

In [19]:
def compute_pass_at_1_from_graded(tasks):
    """
    Try different ways to compute pass@1 and print all.
    """
    n = len(tasks)
    if n == 0:
        return {}
    
    # Way 1: mean of task["pass@1"]
    way1 = sum(t["pass@1"] for t in tasks) / n
    
    # Way 2: mean of max(graded_list) per task
    way2 = sum(
        (1.0 if any(t["graded_list"]) else 0.0) 
        for t in tasks
    ) / n
    
    # Way 3: mean of mean(graded_list) per task
    way3 = sum(
        (sum(t["graded_list"]) / len(t["graded_list"]) if t["graded_list"] else 0.0)
        for t in tasks
    ) / n
    
    # print(f"mean of task['pass@1']:       {way1:.6f}")
    # print(f"mean of any(graded_list):      {way2:.6f}")
    # print(f"mean of mean(graded_list):     {way3:.6f}")
    # print(f"stored pass@1:                 {a['pass@1']:.6f}")
    
    return {"way1": way1, "way2": way2, "way3": way3}



for ind, row in tqdm.tqdm(data[(data['n_rows']==1055)
                              ].sample(10, random_state=42).iterrows()
                         ):
    try:
        a = process_results(row['folder'], row['upfix'])
    except FileNotFoundError:
        print("Not found %s %s"%(row['folder'], row['upfix']))
    
    b = compute_pass_at_1_from_graded(a['tasks'])
    if not b:
        continue
    assert (abs(a['pass@1'] - b['way1']) < 1e-3)
    for c in b.values():
        if not (abs(a['pass@1'] - c) < 1e-3):
            print(ind)
